# Phase 2: Practical Deep Learning & PyTorch Fundamentals
**Topic**: PyTorch Neural Networks, Transfer Learning & Optimization  
**Resource**: [fast.ai — Practical Deep Learning for Coders](https://course.fast.ai/) (Lessons 1–8)

This notebook implements a fine-tuned `ResNet-18` architecture in PyTorch utilizing `AdamW` optimization, cosine annealing learning rate schedules, data transformations, and metric tracking.

In [ ]:
# Install dependencies for Google Colab environment
!pip install torch torchvision scikit-learn matplotlib pillow pandas numpy

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms, models
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute device: {device}')

In [ ]:
# Data Transformation & Tensor Prep
torch.manual_seed(42)
X_train = torch.randn(500, 3, 64, 64)
y_train = torch.randint(0, 10, (500,))
X_test = torch.randn(100, 3, 64, 64)
y_test = torch.randint(0, 10, (100,))

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)
print(f'Train samples: {len(X_train)}, Test samples: {len(X_test)}')

In [ ]:
# Load Pretrained ResNet-18 & Adapt Classification Head
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 10)
)
model = model.to(device)
print(model.fc)

In [ ]:
# Training Loop Setup
criterion = nn.CrossEntropyLoss()
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.AdamW(trainable_params, lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

epochs = 5
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    scheduler.step()
    train_acc = (correct / total) * 100
    print(f'Epoch [{epoch+1}/{epochs}] Loss: {running_loss/total:.4f} | Acc: {train_acc:.2f}%')